In [1]:
import pandas as pd
import sqlite3

# Veritabanı bağlantısı
conn = sqlite3.connect('sqlite-sakila.db') 

# KURAL: Sadece tabloları ham haliyle çekiyoruz 
actor_df = pd.read_sql_query("SELECT * FROM actor", conn)
film_df = pd.read_sql_query("SELECT * FROM film", conn)
film_actor_df = pd.read_sql_query("SELECT * FROM film_actor", conn)
inventory_df = pd.read_sql_query("SELECT * FROM inventory", conn)
rental_df = pd.read_sql_query("SELECT * FROM rental", conn)
payment_df = pd.read_sql_query("SELECT * FROM payment", conn)
customer_df = pd.read_sql_query("SELECT * FROM customer", conn)
category_df = pd.read_sql_query("SELECT * FROM category", conn)
film_category_df = pd.read_sql_query("SELECT * FROM film_category", conn)
address_df = pd.read_sql_query("SELECT * FROM address", conn)
city_df = pd.read_sql_query("SELECT * FROM city", conn)
store_df = pd.read_sql_query("SELECT * FROM store", conn)
staff_df = pd.read_sql_query("SELECT * FROM staff", conn)

print("Tüm tablolar DataFrame olarak yüklendi.")

Tüm tablolar DataFrame olarak yüklendi.


In [2]:
# 1.soru
print(rental_df)

       rental_id              rental_date  inventory_id  customer_id  \
0              1  2005-05-24 22:53:30.000           367          130   
1              2  2005-05-24 22:54:33.000          1525          459   
2              3  2005-05-24 23:03:39.000          1711          408   
3              4  2005-05-24 23:04:41.000          2452          333   
4              5  2005-05-24 23:05:21.000          2079          222   
...          ...                      ...           ...          ...   
16039      16045  2005-08-23 22:25:26.000           772           14   
16040      16046  2005-08-23 22:26:47.000          4364           74   
16041      16047  2005-08-23 22:42:48.000          2088          114   
16042      16048  2005-08-23 22:43:07.000          2019          103   
16043      16049  2005-08-23 22:50:12.000          2666          393   

                   return_date  staff_id          last_update  
0      2005-05-26 22:04:30.000         1  2021-03-06 15:53:41  
1      

In [3]:
# 2.Soru
merged_step1 = pd.merge(film_df, film_actor_df, on='film_id', how='inner')

final_merge = pd.merge(merged_step1, actor_df, on='actor_id', how='inner')

result_q2 = final_merge[['title', 'first_name', 'last_name']]
print(result_q2)

                      title first_name last_name
0          ACADEMY DINOSAUR   PENELOPE   GUINESS
1      ANACONDA CONFESSIONS   PENELOPE   GUINESS
2               ANGELS LIFE   PENELOPE   GUINESS
3     BULWORTH COMMANDMENTS   PENELOPE   GUINESS
4             CHEAPER CLYDE   PENELOPE   GUINESS
...                     ...        ...       ...
5457          SCALAWAG DUCK  CHRISTIAN    NEESON
5458       SENSIBILITY REAR  CHRISTIAN    NEESON
5459    SPIRITED CASUALTIES  CHRISTIAN    NEESON
5460            SPLASH GUMP  CHRISTIAN    NEESON
5461     WORLD LEATHERNECKS  CHRISTIAN    NEESON

[5462 rows x 3 columns]


In [4]:
# Soru 3
merged_film_actor = pd.merge(film_df, film_actor_df, on='film_id', how='inner')

result_q3 = merged_film_actor.groupby('title')['actor_id'].count().reset_index()
result_q3.columns = ['title', 'actor_count']
print(result_q3)

                 title  actor_count
0     ACADEMY DINOSAUR           10
1       ACE GOLDFINGER            4
2     ADAPTATION HOLES            5
3     AFFAIR PREJUDICE            5
4          AFRICAN EGG            5
..                 ...          ...
992     YOUNG LANGUAGE            5
993         YOUTH KICK            5
994       ZHIVAGO CORE            6
995  ZOOLANDER FICTION            5
996          ZORRO ARK            3

[997 rows x 2 columns]


In [5]:
# Soru 4 
merged_q4 = pd.merge(actor_df, film_actor_df, on='actor_id', how='inner')

result_q4 = merged_q4.groupby(['first_name', 'last_name'])['film_id'].count().reset_index()
result_q4.rename(columns={'film_id': 'movie_count'}, inplace=True)

print(result_q4)

    first_name  last_name  movie_count
0         ADAM      GRANT           18
1         ADAM     HOPPER           22
2           AL    GARLAND           26
3         ALAN   DREYFUSS           27
4       ALBERT  JOHANSSON           33
..         ...        ...          ...
194       WILL     WILSON           31
195    WILLIAM    HACKMAN           27
196      WOODY    HOFFMAN           31
197      WOODY      JOLIE           31
198       ZERO       CAGE           25

[199 rows x 3 columns]


In [6]:
# Soru 5
inventory_ids = inventory_df['film_id'].unique()

not_in_inventory = film_df[~film_df['film_id'].isin(inventory_ids)]

print(f"Envanterde olmayan film sayısı: {len(not_in_inventory)}")

Envanterde olmayan film sayısı: 42


In [7]:
# Soru 6
f_cols = film_df[['film_id', 'title']]
i_cols = inventory_df[['inventory_id', 'film_id']]
r_cols = rental_df[['rental_id', 'inventory_id']]
p_cols = payment_df[['rental_id', 'amount']]

m1 = pd.merge(f_cols, i_cols, on='film_id')
m2 = pd.merge(m1, r_cols, on='inventory_id')
final_join = pd.merge(m2, p_cols, on='rental_id')

result_q6 = final_join.groupby('title').agg(
    rental_count=('rental_id', 'count'),
    total_revenue=('amount', 'sum')
).reset_index()

print(result_q6)

                 title  rental_count  total_revenue
0     ACADEMY DINOSAUR            23          36.77
1       ACE GOLDFINGER             7          52.93
2     ADAPTATION HOLES            12          37.88
3     AFFAIR PREJUDICE            23          91.77
4          AFRICAN EGG            12          51.88
..                 ...           ...            ...
953     YOUNG LANGUAGE             7           6.93
954         YOUTH KICK             6          16.94
955       ZHIVAGO CORE             9          14.91
956  ZOOLANDER FICTION            17          73.83
957          ZORRO ARK            31         214.69

[958 rows x 3 columns]


In [8]:
# Soru 7
inventory_ids = inventory_df['film_id'].unique()
missing_films = film_df[~film_df['film_id'].isin(inventory_ids)]

result_q7 = missing_films[['title', 'rental_rate']]

print(result_q7)

                      title  rental_rate
13           ALICE FANTASIA         0.99
32              APOLLO TEEN         2.99
35           ARGONAUTS TOWN         0.99
37            ARK RIDGEMONT         0.99
40     ARSENIC INDEPENDENCE         0.99
86        BOONDOCK BALLROOM         0.99
107           BUTCH PANTHER         0.99
127           CATCH AMISTAD         0.99
143     CHINATOWN GLADIATOR         4.99
147          CHOCOLATE DUCK         2.99
170    COMMANDMENTS EXPRESS         4.99
191        CROSSING DIVORCE         4.99
194         CROWDS TELEMARK         4.99
197        CRYSTAL BREAKING         2.99
216              DAZED PUNK         4.99
220  DELIVERANCE MULHOLLAND         0.99
317       FIREHOUSE VIETNAM         0.99
324           FLOATS GARDEN         2.99
331   FRANKENSTEIN STRANGER         0.99
358      GLADIATOR WESTWARD         4.99
385               GUMP DATE         4.99
403           HATE HANDICAP         0.99
418             HOCUS FRIDA         2.99
494        KENTU

In [9]:
# Soru 8
not_returned = rental_df[rental_df['return_date'].isnull()]

cust_counts = not_returned.groupby('customer_id').size()

count_q8 = len(cust_counts[cust_counts > 1])

print(f"Birden fazla DVD iade etmeyen müşteri sayısı: {count_q8}")

Birden fazla DVD iade etmeyen müşteri sayısı: 23


In [10]:
# Soru 9
merged_q9 = pd.merge(customer_df, rental_df, on='customer_id')

result_q9 = merged_q9.groupby(['first_name', 'last_name'])['rental_id'].count().reset_index()

print(result_q9)

    first_name last_name  rental_id
0        AARON     SELBY         24
1         ADAM     GOOCH         22
2       ADRIAN     CLARY         19
3        AGNES    BISHOP         23
4         ALAN      KAHN         26
..         ...       ...        ...
594     WILLIE   MARKHAM         25
595      WILMA  RICHARDS         20
596    YOLANDA    WEAVER         27
597     YVONNE   WATKINS         21
598    ZACHARY      HITE         31

[599 rows x 3 columns]


In [11]:
# Soru 10
f_cols = film_df[['film_id', 'title']]
fc_cols = film_category_df[['film_id', 'category_id']]
c_cols = category_df[['category_id', 'name']]
i_cols = inventory_df[['inventory_id', 'film_id']]
r_cols = rental_df[['rental_id', 'inventory_id']]
p_cols = payment_df[['rental_id', 'amount']]

m1 = pd.merge(f_cols, fc_cols, on='film_id')
m2 = pd.merge(m1, c_cols, on='category_id')
m3 = pd.merge(m2, i_cols, on='film_id')
m4 = pd.merge(m3, r_cols, on='inventory_id')
final_q10 = pd.merge(m4, p_cols, on='rental_id')

result_q10 = final_q10.groupby(['name', 'title']).agg(
    rental_count=('rental_id', 'count'),
    total_revenue=('amount', 'sum')
).reset_index().sort_values(by='rental_count', ascending=False)

print(result_q10)

            name                title  rental_count  total_revenue
909       Travel   BUCKET BROTHERHOOD            34         180.66
536      Foreign     ROCKETEER MOTHER            33         116.67
757          New  RIDGEMONT SUBMARINE            32         130.68
572        Games       GRIT CLOCKWORK            32         110.68
93     Animation       JUGGLER HARDLY            32          96.68
..           ...                  ...           ...            ...
513      Foreign      INFORMER DOUBLE             5          27.95
570        Games          GLORY TRACY             5          14.95
656       Horror          TRAIN BUNCH             4          24.96
525      Foreign          MIXED DOORS             4          15.96
315  Documentary       HARDLY ROBBERS             4          15.96

[958 rows x 4 columns]


In [12]:
# Soru 11
c_cols = category_df[['category_id', 'name']]
fc_cols = film_category_df[['category_id', 'film_id']]
i_cols = inventory_df[['inventory_id', 'film_id']]
r_cols = rental_df[['rental_id', 'inventory_id', 'rental_date']]


m1 = pd.merge(c_cols, fc_cols, on='category_id')
m2 = pd.merge(m1, i_cols, on='film_id')
final_df = pd.merge(m2, r_cols, on='inventory_id')

final_df['rental_date'] = pd.to_datetime(final_df['rental_date'])
final_df['date_only'] = final_df['rental_date'].dt.date

result_q11 = final_df.groupby(['name', 'date_only'])['rental_id'].count().reset_index()
result_q11.columns = ['genre', 'rental_date', 'count']

print(result_q11)

      genre rental_date  count
0    Action  2005-05-25     10
1    Action  2005-05-26     15
2    Action  2005-05-27      9
3    Action  2005-05-28     20
4    Action  2005-05-29     12
..      ...         ...    ...
628  Travel  2005-08-20     29
629  Travel  2005-08-21     39
630  Travel  2005-08-22     21
631  Travel  2005-08-23     27
632  Travel  2006-02-14     10

[633 rows x 3 columns]


In [13]:
# Soru 12
c_cols = category_df[['category_id', 'name']]
fc_cols = film_category_df[['category_id', 'film_id']]
f_cols = film_df[['film_id', 'title']]
i_cols = inventory_df[['inventory_id', 'film_id']]
r_cols = rental_df[['rental_id', 'inventory_id']]

m1 = pd.merge(c_cols, fc_cols, on='category_id')
m2 = pd.merge(m1, f_cols, on='film_id')      
m3 = pd.merge(m2, i_cols, on='film_id')
final_df = pd.merge(m3, r_cols, on='inventory_id')

result_q12 = final_df.groupby(['name', 'title'])['rental_id'].count().reset_index()
result_q12.columns = ['genre', 'film_title', 'rental_count']

result_q12 = result_q12.sort_values(by='rental_count', ascending=False)

print(result_q12)

           genre           film_title  rental_count
909       Travel   BUCKET BROTHERHOOD            34
536      Foreign     ROCKETEER MOTHER            33
757          New  RIDGEMONT SUBMARINE            32
572        Games       GRIT CLOCKWORK            32
93     Animation       JUGGLER HARDLY            32
..           ...                  ...           ...
513      Foreign      INFORMER DOUBLE             5
570        Games          GLORY TRACY             5
656       Horror          TRAIN BUNCH             4
525      Foreign          MIXED DOORS             4
315  Documentary       HARDLY ROBBERS             4

[958 rows x 3 columns]


In [14]:
# Soru 13
f_cols = film_df[['film_id', 'title']]
i_cols = inventory_df[['inventory_id', 'film_id']]
# rental_date lazım (hesap için), return_date lazım (filtre için)
r_cols = rental_df[['inventory_id', 'rental_date', 'return_date']] 

m1 = pd.merge(f_cols, i_cols, on='film_id')
final_df = pd.merge(m1, r_cols, on='inventory_id')

on_rent = final_df[final_df['return_date'].isnull()].copy()

now = pd.Timestamp.now()
on_rent['rental_date'] = pd.to_datetime(on_rent['rental_date'])
on_rent['days_on_shelf'] = (now - on_rent['rental_date']).dt.days

result_q13 = on_rent[['title', 'days_on_shelf']].sort_values(by='days_on_shelf', ascending=False)

print(result_q13)

                       title  days_on_shelf
16          ACADEMY DINOSAUR           7415
9691         MUMMY CREATURES           7238
10016           NONE SPIKING           7238
10187    OPERATION OPERATION           7238
10223      OPPOSITE NECKLACE           7238
...                      ...            ...
4783            FAMILY SWEET           7238
5029        FIGHT JAWBREAKER           7238
5210             FLYING HOOK           7238
5291   FORRESTER COMANCHEROS           7238
15995           ZHIVAGO CORE           7238

[183 rows x 2 columns]


In [15]:
# Soru 14
f_cols = film_df[['film_id', 'rental_duration']] 
i_cols = inventory_df[['inventory_id', 'film_id']]
r_cols = rental_df[['rental_id', 'inventory_id', 'rental_date', 'return_date']]

m1 = pd.merge(r_cols, i_cols, on='inventory_id')
final_df = pd.merge(m1, f_cols, on='film_id')

final_df['rental_date'] = pd.to_datetime(final_df['rental_date'])
final_df['return_date'] = pd.to_datetime(final_df['return_date'])

final_df['due_date'] = final_df['rental_date'] + pd.to_timedelta(final_df['rental_duration'], unit='D')

def get_status(row):
    if pd.isnull(row['return_date']):
        return 'Not Returned'
    elif row['return_date'] > row['due_date']:
        return 'Late'
    elif row['return_date'] < row['due_date']:
        return 'Early'
    else:
        return 'On Time'

final_df['status'] = final_df.apply(get_status, axis=1)

result_q14 = final_df['status'].value_counts()

print(result_q14)

status
Late            8121
Early           7738
Not Returned     183
On Time            2
Name: count, dtype: int64


In [16]:
# Soru 15
c_cols = customer_df[['customer_id', 'first_name', 'last_name']]
r_cols = rental_df[['rental_id', 'customer_id']]

merged_df = pd.merge(c_cols, r_cols, on='customer_id')

counts = merged_df.groupby(['first_name', 'last_name'])['rental_id'].count().reset_index()
counts.rename(columns={'rental_id': 'rental_count'}, inplace=True)

top_customer = counts.sort_values(by='rental_count', ascending=False).iloc[0]

print(f"En çok kiralayan müşteri: {top_customer['first_name']} {top_customer['last_name']} ({top_customer['rental_count']} adet)")

En çok kiralayan müşteri: ELEANOR HUNT (46 adet)


In [17]:
# Soru 16
c_cols = category_df[['category_id', 'name']]
fc_cols = film_category_df[['category_id', 'film_id']]
i_cols = inventory_df[['inventory_id', 'film_id']]
r_cols = rental_df[['rental_id', 'inventory_id']]

m1 = pd.merge(c_cols, fc_cols, on='category_id')
m2 = pd.merge(m1, i_cols, on='film_id') 
merged_df = pd.merge(m2, r_cols, on='inventory_id')

res_16 = merged_df.groupby('name')['rental_id'].count().reset_index()
res_16 = res_16.sort_values('rental_id', ascending=False).head(1)
print(res_16)

      name  rental_id
14  Sports       1179


In [18]:
# Soru 17
s_cols = staff_df[['staff_id', 'first_name', 'last_name']]
r_cols = rental_df[['rental_id', 'staff_id']]

merged_df = pd.merge(s_cols, r_cols, on='staff_id')

res_17 = merged_df.groupby(['first_name', 'last_name'])['rental_id'].count().reset_index()
res_17 = res_17.sort_values('rental_id', ascending=False).head(1)
print(res_17)

  first_name last_name  rental_id
1       Mike   Hillyer       8040


In [19]:
# Soru 18
f_cols = film_df[['film_id', 'title']]
i_cols = inventory_df[['inventory_id', 'film_id']]
r_cols = rental_df[['rental_id', 'inventory_id']]
p_cols = payment_df[['rental_id', 'amount']]

m1 = pd.merge(f_cols, i_cols, on='film_id')
m2 = pd.merge(m1, r_cols, on='inventory_id')
merged_df = pd.merge(m2, p_cols, on='rental_id')

res_18 = merged_df.groupby('title')['amount'].sum().reset_index()
res_18 = res_18.sort_values('amount', ascending=False).head(1)
print(res_18)

                title  amount
841  TELEGRAPH VOYAGE  231.73


In [20]:
# Soru 19
c_cols = customer_df[['customer_id', 'first_name', 'last_name']]
p_cols = payment_df[['customer_id', 'amount']]

merged_df = pd.merge(c_cols, p_cols, on='customer_id')

res_19 = merged_df.groupby(['first_name', 'last_name'])['amount'].sum().reset_index()
res_19 = res_19.sort_values('amount', ascending=False)
print(res_19.head)

<bound method NDFrame.head of     first_name last_name  amount
318       KARL      SEAL  221.55
175    ELEANOR      HUNT  216.54
105      CLARA      SHAW  195.58
474     RHONDA   KENNEDY  194.61
389     MARION    SNYDER  194.61
..         ...       ...     ...
33       ANNIE   RUSSELL   58.82
296     JOHNNY    TURPIN   57.81
71       BRIAN     WYMAN   52.88
351      LEONA    OBRIEN   50.86
83    CAROLINE    BOWMAN   50.85

[599 rows x 3 columns]>


In [21]:
# Soru 20
c_cols = category_df[['category_id', 'name']]
fc_cols = film_category_df[['category_id', 'film_id']]
i_cols = inventory_df[['inventory_id', 'film_id']]
r_cols = rental_df[['rental_id', 'inventory_id']]
p_cols = payment_df[['rental_id', 'amount']]

m1 = pd.merge(c_cols, fc_cols, on='category_id')
m2 = pd.merge(m1, i_cols, on='film_id')
m3 = pd.merge(m2, r_cols, on='inventory_id')
merged_df = pd.merge(m3, p_cols, on='rental_id')

res_20 = merged_df.groupby('name').agg(
    rental_count=('rental_id', 'count'),
    revenue=('amount', 'sum')
).reset_index()
print(res_20)

           name  rental_count  revenue
0        Action          1112  4375.85
1     Animation          1166  4656.30
2      Children           945  3655.55
3      Classics           939  3639.59
4        Comedy           941  4383.58
5   Documentary          1050  4217.52
6         Drama          1060  4587.39
7        Family          1096  4226.07
8       Foreign          1033  4270.67
9         Games           969  4281.33
10       Horror           846  3722.54
11        Music           830  3417.72
12          New           940  4351.62
13       Sci-Fi          1101  4756.98
14       Sports          1179  5314.21
15       Travel           837  3549.64


In [22]:
# Soru 21
f_cols = film_df[['film_id', 'title']]
i_cols = inventory_df[['inventory_id', 'film_id']]
r_cols = rental_df[['rental_id', 'inventory_id', 'rental_date', 'return_date']]

m1 = pd.merge(f_cols, i_cols, on='film_id')
merged_df = pd.merge(m1, r_cols, on='inventory_id')

merged_df['rental_date'] = pd.to_datetime(merged_df['rental_date'])
merged_df['return_date'] = pd.to_datetime(merged_df['return_date'])
merged_df['duration'] = (merged_df['return_date'] - merged_df['rental_date']).dt.total_seconds() / (24*3600)

res_21 = merged_df.groupby('title')['duration'].max().reset_index()
res_21 = res_21.sort_values('duration', ascending=False).head(5)
print(res_21)

                  title  duration
400  HOLOCAUST HIGHBALL  9.249306
393     HIGHBALL POTTER  9.249306
160   CONEHEADS SMOOCHY  9.248611
625          PANIC CLUB  9.248611
535          MASK PEACH  9.248611


In [23]:
# Soru 22
f_cols = film_df[['film_id', 'title']]
i_cols = inventory_df[['inventory_id', 'film_id']]
r_cols = rental_df[['rental_id', 'inventory_id']]

m1 = pd.merge(f_cols, i_cols, on='film_id')
merged_df = pd.merge(m1, r_cols, on='inventory_id')

res_22 = merged_df.groupby('title')['rental_id'].count().reset_index()
res_22 = res_22.sort_values('rental_id', ascending=True).head(5)
print(res_22)

                  title  rental_id
558         MIXED DOORS          4
866         TRAIN BUNCH          4
378      HARDLY ROBBERS          4
585  MUSSOLINI SPOILERS          5
315   FREEDOM CLEOPATRA          5


In [24]:
# Soru 24
c_cols = customer_df[['customer_id', 'first_name', 'last_name']]
p_cols = payment_df[['customer_id', 'amount']]

merged_df = pd.merge(c_cols, p_cols, on='customer_id')

res_24 = merged_df.groupby(['first_name', 'last_name'])['amount'].sum().reset_index()
res_24 = res_24.sort_values('amount', ascending=False).head(5)
print(res_24)

    first_name last_name  amount
318       KARL      SEAL  221.55
175    ELEANOR      HUNT  216.54
105      CLARA      SHAW  195.58
474     RHONDA   KENNEDY  194.61
389     MARION    SNYDER  194.61


In [25]:
# Soru 25
f_cols = film_df[['film_id', 'title']]
i_cols = inventory_df[['inventory_id', 'film_id']]
r_cols = rental_df[['rental_id', 'inventory_id', 'rental_date', 'return_date']]

m1 = pd.merge(f_cols, i_cols, on='film_id')
df = pd.merge(m1, r_cols, on='inventory_id')
df['rental_date'] = pd.to_datetime(df['rental_date'])
df['return_date'] = pd.to_datetime(df['return_date'])
df['duration'] = (df['return_date'] - df['rental_date']).dt.total_seconds() / (3600*24)

res_25 = df.groupby('title')['duration'].mean().reset_index()
print(res_25)

                 title  duration
0     ACADEMY DINOSAUR  4.997128
1       ACE GOLDFINGER  5.642708
2     ADAPTATION HOLES  3.465625
3     AFFAIR PREJUDICE  4.758333
4          AFRICAN EGG  7.106629
..                 ...       ...
953     YOUNG LANGUAGE  4.624405
954         YOUTH KICK  5.562963
955       ZHIVAGO CORE  5.950087
956  ZOOLANDER FICTION  5.542647
957          ZORRO ARK  4.539180

[958 rows x 2 columns]


In [26]:
# Soru 26
c_cols = category_df[['category_id', 'name']]
fc_cols = film_category_df[['category_id', 'film_id']]
f_cols = film_df[['film_id', 'title']]
i_cols = inventory_df[['inventory_id', 'film_id']]
r_cols = rental_df[['rental_id', 'inventory_id']]

m1 = pd.merge(c_cols, fc_cols, on='category_id')
m2 = pd.merge(m1, f_cols, on='film_id')
m3 = pd.merge(m2, i_cols, on='film_id')
df = pd.merge(m3, r_cols, on='inventory_id')

counts = df.groupby(['name', 'title'])['rental_id'].count().reset_index(name='count')

res_26 = counts.groupby('name').apply(lambda x: x.nlargest(1, 'count')).reset_index(drop=True)
print(res_26)

           name                title  count
0        Action  RUGRATS SHAKESPEARE     30
1     Animation       JUGGLER HARDLY     32
2      Children         ROBBERS JOON     31
3      Classics       TIMBERLAND SKY     31
4        Comedy            ZORRO ARK     31
5   Documentary            WIFE TURN     31
6         Drama         HOBBIT ALIEN     31
7        Family        APACHE DIVINE     31
8       Foreign     ROCKETEER MOTHER     33
9         Games       FORWARD TEMPLE     32
10       Horror         PULP BEVERLY     30
11        Music        SCALAWAG DUCK     32
12          New  RIDGEMONT SUBMARINE     32
13       Sci-Fi    GOODFELLAS SALUTE     31
14       Sports  GLEAMING JAWBREAKER     29
15       Travel   BUCKET BROTHERHOOD     34


In [27]:
# Soru 28
c_cols = customer_df[['customer_id', 'first_name', 'last_name']]
r_cols = rental_df[['rental_id', 'customer_id', 'return_date']]

merged_df = pd.merge(c_cols, r_cols, on='customer_id')

# Filtreleme: Return date boş olanlar
unreturned = merged_df[merged_df['return_date'].isnull()]

res_28 = unreturned.groupby(['first_name', 'last_name'])['rental_id'].count().reset_index()
res_28 = res_28.sort_values('rental_id', ascending=False).head(1)
print(res_28)

    first_name last_name  rental_id
144      TAMMY   SANDERS          3


In [28]:
# Soru 28
c_cols = customer_df[['customer_id', 'first_name', 'last_name', 'store_id']]
r_cols = rental_df[['rental_id', 'customer_id']]

merged_df = pd.merge(c_cols, r_cols, on='customer_id')

res_30 = merged_df.groupby(['first_name', 'last_name', 'store_id'])['rental_id'].count().reset_index()
res_30 = res_30.sort_values('rental_id', ascending=False).head(5)
print(res_30)

    first_name last_name  store_id  rental_id
175    ELEANOR      HUNT         1         46
318       KARL      SEAL         2         45
379     MARCIA      DEAN         1         42
105      CLARA      SHAW         1         42
536      TAMMY   SANDERS         2         41


In [29]:
# Soru 27
c_cols = category_df[['category_id', 'name']]
fc_cols = film_category_df[['category_id', 'film_id']]
f_cols = film_df[['film_id', 'title']]
i_cols = inventory_df[['inventory_id', 'film_id']]
r_cols = rental_df[['rental_id', 'inventory_id']]
p_cols = payment_df[['rental_id', 'amount']]

m1 = pd.merge(c_cols, fc_cols, on='category_id')
m2 = pd.merge(m1, f_cols, on='film_id')
m3 = pd.merge(m2, i_cols, on='film_id')
m4 = pd.merge(m3, r_cols, on='inventory_id')
final_df = pd.merge(m4, p_cols, on='rental_id')

res_27 = final_df.groupby(['name', 'title'])['amount'].sum().reset_index()
res_27 = res_27.sort_values('amount', ascending=False).head(1)
print(res_27)

      name             title  amount
705  Music  TELEGRAPH VOYAGE  231.73


In [30]:
# Soru 29
s_cols = staff_df[['staff_id', 'first_name', 'last_name']]
r_cols = rental_df[['rental_id', 'staff_id']]

merged_df = pd.merge(s_cols, r_cols, on='staff_id')

res_29 = merged_df.groupby(['first_name', 'last_name'])['rental_id'].count().reset_index()
res_29 = res_29.sort_values('rental_id', ascending=False).head(5)
print(res_29)

  first_name last_name  rental_id
1       Mike   Hillyer       8040
0        Jon  Stephens       8004


In [31]:
# Soru 31
c_cols = category_df[['category_id', 'name']]
fc_cols = film_category_df[['category_id', 'film_id']]
f_cols = film_df[['film_id', 'title']]
i_cols = inventory_df[['inventory_id', 'film_id']]
r_cols = rental_df[['rental_id', 'inventory_id']]


m1 = pd.merge(c_cols, fc_cols, on='category_id')
m2 = pd.merge(m1, f_cols, on='film_id')
m3 = pd.merge(m2, i_cols, on='film_id')
final_df = pd.merge(m3, r_cols, on='inventory_id')

res_31 = final_df.groupby(['name', 'title'])['rental_id'].count().reset_index(name='rental_count')
res_31 = res_31.sort_values('rental_count', ascending=True).head(1)
print(res_31)

            name           title  rental_count
315  Documentary  HARDLY ROBBERS             4


In [36]:
# Soru 32
city_cols = city_df[['city_id', 'city']]
addr_cols = address_df[['address_id', 'city_id']]
cust_cols = customer_df[['customer_id', 'address_id', 'first_name', 'last_name']]
rent_cols = rental_df[['rental_id', 'customer_id']]

m1 = pd.merge(city_cols, addr_cols, on='city_id')
m2 = pd.merge(m1, cust_cols, on='address_id')
final_df = pd.merge(m2, rent_cols, on='customer_id')

res_32 = final_df.groupby(['city', 'first_name', 'last_name'])['rental_id'].count().reset_index(name='rental_count')
res_32 = res_32.sort_values('rental_count', ascending=False).head(5)
print(res_32)

            city first_name last_name  rental_count
433  Saint-Denis    ELEANOR      HUNT            46
96    Cape Coral       KARL      SEAL            45
333    Molodetno      CLARA      SHAW            42
518        Tanza     MARCIA      DEAN            42
103     Changhwa      TAMMY   SANDERS            41


In [37]:
# Soru 33
p_cols = payment_df[['rental_id', 'amount']]

full_df = pd.merge(final_df, p_cols, on='rental_id')

res_33 = full_df.groupby(['city', 'first_name', 'last_name'])['amount'].sum().reset_index(name='total_spent')
res_33 = res_33.sort_values('total_spent', ascending=False).head(5)
print(res_33)

                    city first_name last_name  total_spent
96            Cape Coral       KARL      SEAL       221.55
433          Saint-Denis    ELEANOR      HUNT       216.54
333            Molodetno      CLARA      SHAW       195.58
23             Apeldoorn     RHONDA   KENNEDY       194.61
447  Santa Brbara dOeste     MARION    SNYDER       194.61


In [32]:
# Soru 34
city_cols = city_df[['city_id', 'city']]
addr_cols = address_df[['address_id', 'city_id']]
cust_cols = customer_df[['customer_id', 'address_id']]
rent_cols = rental_df[['rental_id', 'customer_id', 'inventory_id']]
inv_cols = inventory_df[['inventory_id', 'film_id']]
film_cols = film_df[['film_id', 'title']]

m1 = pd.merge(city_cols, addr_cols, on='city_id')
m2 = pd.merge(m1, cust_cols, on='address_id')
m3 = pd.merge(m2, rent_cols, on='customer_id')
m4 = pd.merge(m3, inv_cols, on='inventory_id')
df = pd.merge(m4, film_cols, on='film_id')

res_34 = df.groupby(['city', 'title'])['rental_id'].count().reset_index()
res_34 = res_34.sort_values('rental_id', ascending=False).head(5)
print(res_34)

           city              title  rental_id
14182   Trshavn  FLATLINERS KILLER          3
12735  Sorocaba    CADDYSHACK JEDI          3
7308     Kurgan   DETECTIVE VISION          3
7828       Lima    DISCIPLE MOTHER          3
2598    Caracas   AFFAIR PREJUDICE          2


In [38]:
# Soru 35
i_cols = inventory_df[['inventory_id', 'film_id']]
f_cols = film_df[['film_id', 'title']]
rent_cols_w_inv = rental_df[['rental_id', 'customer_id', 'inventory_id']]

m1 = pd.merge(city_df[['city_id','city']], address_df[['address_id','city_id']], on='city_id')
m2 = pd.merge(m1, customer_df[['customer_id','address_id']], on='address_id')
m3 = pd.merge(m2, rent_cols_w_inv, on='customer_id')
m4 = pd.merge(m3, i_cols, on='inventory_id')
complete_df = pd.merge(m4, f_cols, on='film_id')

res_35 = complete_df.groupby(['city', 'title'])['rental_id'].count().reset_index(name='rental_count')
res_35 = res_35.sort_values('rental_count', ascending=True).head(5)
print(res_35)

                     city            title  rental_count
0      A Corua (La Corua)     BLADE POLISH             1
10513          Phnom Penh  TOURIST PELICAN             1
10514          Phnom Penh  TWISTED PIRATES             1
10515          Phnom Penh   VOYAGE LEGALLY             1
10516          Phnom Penh        WIFE TURN             1


In [41]:
# Soru 36
city_cols = city_df[['city_id', 'city']]
addr_cols = address_df[['address_id', 'city_id']]
cust_cols = customer_df[['customer_id', 'address_id']]
rent_cols = rental_df[['rental_id', 'customer_id', 'inventory_id']]
inv_cols = inventory_df[['inventory_id', 'film_id']]
film_cols = film_df[['film_id', 'title']]
pay_cols = payment_df[['rental_id', 'amount']]

m1 = pd.merge(city_cols, addr_cols, on='city_id')
m2 = pd.merge(m1, cust_cols, on='address_id')
m3 = pd.merge(m2, rent_cols, on='customer_id')
m4 = pd.merge(m3, inv_cols, on='inventory_id')
m5 = pd.merge(m4, film_cols, on='film_id')

final_df = pd.merge(m5, pay_cols, on='rental_id')

result_q36 = final_df.groupby(['city', 'title'])['amount'].sum().reset_index(name='total_revenue')

result_q36 = result_q36.sort_values(by='total_revenue', ascending=False).head(5)

print(result_q36)

                      city             title  total_revenue
10752          Probolinggo      MINDS TRUMAN          18.98
10565                Plock  CALIFORNIA BIRDS          18.98
6650                Kamyin    ROSES TREASURE          17.98
11854  Santa Brbara dOeste         WIFE TURN          17.98
3520                Datong      EAGLES PANKY          16.98


In [42]:
# Soru 37
res_37 = res_36.sort_values('total_revenue', ascending=True).head(5)
print(res_37)

                      city             title  total_revenue
3520                Datong      EAGLES PANKY          16.98
6650                Kamyin    ROSES TREASURE          17.98
11854  Santa Brbara dOeste         WIFE TURN          17.98
10752          Probolinggo      MINDS TRUMAN          18.98
10565                Plock  CALIFORNIA BIRDS          18.98


In [39]:
# Soru 38
top_customer = rental_df.groupby('customer_id').size().sort_values(ascending=False).index[0]

cust_cols = customer_df[['customer_id', 'first_name', 'last_name']]
rent_cols = rental_df[['rental_id', 'customer_id', 'inventory_id']]
inv_cols = inventory_df[['inventory_id', 'film_id']]
film_cols = film_df[['film_id', 'title']]

target_rentals = rent_cols[rent_cols['customer_id'] == top_customer]

m1 = pd.merge(target_rentals, cust_cols, on='customer_id')
m2 = pd.merge(m1, inv_cols, on='inventory_id')
final_df = pd.merge(m2, film_cols, on='film_id')

res_38 = final_df[['first_name', 'last_name', 'title']]
print(res_38.head(5))

  first_name last_name                title
0    ELEANOR      HUNT   PREJUDICE OLEANDER
1    ELEANOR      HUNT           GUN BONNIE
2    ELEANOR      HUNT  SNATCHERS MONTEZUMA
3    ELEANOR      HUNT     ENGLISH BULWORTH
4    ELEANOR      HUNT       FORWARD TEMPLE


In [34]:
# Soru 40
top_paying_cust = payment_df.groupby('customer_id')['amount'].sum().sort_values(ascending=False).index[0]

rent_cols = rental_df[['rental_id', 'customer_id', 'inventory_id']]
inv_cols = inventory_df[['inventory_id', 'film_id']]
film_cols = film_df[['film_id', 'title']]

target_rentals = rent_cols[rent_cols['customer_id'] == top_paying_cust]

m1 = pd.merge(target_rentals, inv_cols, on='inventory_id')
res_40 = pd.merge(m1, film_cols, on='film_id')

print(res_40[['title']])

                    title
0        DESTINY SATURDAY
1          CYCLONE FAMILY
2              SLUMS DUCK
3          FIDELITY DEVIL
4             SPLASH GUMP
5       MISSION ZOOLANDER
6        MULHOLLAND BEAST
7          PRINCESS GIANT
8           PARIS WEEKEND
9               RACER EGG
10         WEDDING APOLLO
11         WEDDING APOLLO
12       BIKINI BORROWERS
13         ROBBERY BRIGHT
14               SPY MILE
15        MOONWALKER FOOL
16       METAL ARMAGEDDON
17    RIDGEMONT SUBMARINE
18            HIGH ENCINO
19           DURHAM PANKY
20            GANGS PRIDE
21             DATE SPEED
22     HEAVYWEIGHTS BEAST
23             DUMBO LUST
24        GOLDMINE TYCOON
25       ENGLISH BULWORTH
26    BRINGING HYSTERICAL
27  FORRESTER COMANCHEROS
28        REAP UNFAITHFUL
29           DESIRE ALIEN
30        LONELY ELEPHANT
31          BETRAYED REAR
32       SINNERS ATLANTIS
33               INCH JET
34       FOOL MOCKINGBIRD
35         BLUES INSTINCT
36         REMEMBER DIARY
37          

In [43]:
# Soru 41
low_customer_id = payment_df.groupby('customer_id')['amount'].sum().idxmin()

target_rentals = rental_df[rental_df['customer_id'] == low_customer_id][['inventory_id', 'customer_id']]

m1 = pd.merge(target_rentals, customer_df[['customer_id', 'first_name', 'last_name']], on='customer_id')
m2 = pd.merge(m1, inventory_df[['inventory_id', 'film_id']], on='inventory_id')
res_41 = pd.merge(m2, film_df[['film_id', 'title']], on='film_id')

print(res_41[['first_name', 'last_name', 'title']])

   first_name last_name                   title
0    CAROLINE    BOWMAN          FRENCH HOLIDAY
1    CAROLINE    BOWMAN         WHISPERER GIANT
2    CAROLINE    BOWMAN              MASK PEACH
3    CAROLINE    BOWMAN         ARMAGEDDON LOST
4    CAROLINE    BOWMAN         ILLUSION AMELIE
5    CAROLINE    BOWMAN       HOOSIERS BIRDCAGE
6    CAROLINE    BOWMAN        CAMELOT VACATION
7    CAROLINE    BOWMAN              JADE BUNCH
8    CAROLINE    BOWMAN         KILLER INNOCENT
9    CAROLINE    BOWMAN         DISCIPLE MOTHER
10   CAROLINE    BOWMAN       GOODFELLAS SALUTE
11   CAROLINE    BOWMAN        EMPIRE MALKOVICH
12   CAROLINE    BOWMAN  RESURRECTION SILVERADO
13   CAROLINE    BOWMAN             TEXAS WATCH
14   CAROLINE    BOWMAN    IMPOSSIBLE PREJUDICE


In [44]:
# Soru 42
target_rentals = rental_df[rental_df['customer_id'] == low_customer_id][['inventory_id', 'customer_id']]

i_cols = inventory_df[['inventory_id', 'film_id']]
fc_cols = film_category_df[['film_id', 'category_id']]
cat_cols = category_df[['category_id', 'name']]
c_cols = customer_df[['customer_id', 'first_name']]

m1 = pd.merge(target_rentals, i_cols, on='inventory_id')
m2 = pd.merge(m1, fc_cols, on='film_id')
m3 = pd.merge(m2, cat_cols, on='category_id')
full_q42 = pd.merge(m3, c_cols, on='customer_id')

res_42 = full_q42.groupby(['first_name', 'name'])['inventory_id'].count().reset_index(name='count')
res_42 = res_42.sort_values('count', ascending=False).head(1)
print(res_42)

  first_name    name  count
6   CAROLINE  Sci-Fi      4


In [45]:
# Soru 43
rent_inv = pd.merge(rental_df[['rental_id', 'inventory_id']], inventory_df[['inventory_id', 'film_id']], on='inventory_id')
top_film_id = rent_inv['film_id'].value_counts().idxmax()
top_film_title = film_df[film_df['film_id'] == top_film_id]['title'].iloc[0]

target_inv_ids = inventory_df[inventory_df['film_id'] == top_film_id]['inventory_id']
target_rentals = rental_df[rental_df['inventory_id'].isin(target_inv_ids)]

res_43 = pd.merge(target_rentals, staff_df[['staff_id', 'first_name', 'last_name']], on='staff_id')
res_43 = res_43.groupby(['first_name', 'last_name']).size().reset_index(name='count')
res_43['film_title'] = top_film_title
print(res_43)

  first_name last_name  count          film_title
0        Jon  Stephens     18  BUCKET BROTHERHOOD
1       Mike   Hillyer     16  BUCKET BROTHERHOOD


In [46]:
# Soru 44
least_film_id = rent_inv['film_id'].value_counts().idxmin()
least_film_title = film_df[film_df['film_id'] == least_film_id]['title'].iloc[0]

target_inv_ids = inventory_df[inventory_df['film_id'] == least_film_id]['inventory_id']
target_rentals = rental_df[rental_df['inventory_id'].isin(target_inv_ids)]

res_44 = pd.merge(target_rentals, staff_df[['staff_id', 'first_name', 'last_name']], on='staff_id')
res_44 = res_44.groupby(['first_name', 'last_name']).size().reset_index(name='count')
res_44['film_title'] = least_film_title
print(res_44)

  first_name last_name  count   film_title
0        Jon  Stephens      1  MIXED DOORS
1       Mike   Hillyer      3  MIXED DOORS


In [47]:
# Soru 46
m1 = pd.merge(payment_df[['rental_id', 'amount']], rental_df[['rental_id', 'inventory_id']], on='rental_id')
m2 = pd.merge(m1, inventory_df[['inventory_id', 'film_id']], on='inventory_id')
top_rev_film_id = m2.groupby('film_id')['amount'].sum().idxmax()

target_inv_ids = inventory_df[inventory_df['film_id'] == top_rev_film_id]['inventory_id']
target_rentals = rental_df[rental_df['inventory_id'].isin(target_inv_ids)]

m_staff = pd.merge(target_rentals, staff_df[['staff_id', 'first_name']], on='staff_id')
m_pay = pd.merge(m_staff, payment_df[['rental_id', 'amount']], on='rental_id')

res_45 = m_pay.groupby('first_name')['amount'].sum().reset_index(name='total_revenue')
print(res_45)

  first_name  total_revenue
0        Jon         147.83
1       Mike          83.90


In [48]:
# Soru 46
low_rev_film_id = m2.groupby('film_id')['amount'].sum().idxmin()

target_inv_ids = inventory_df[inventory_df['film_id'] == low_rev_film_id]['inventory_id']
target_rentals = rental_df[rental_df['inventory_id'].isin(target_inv_ids)]

m_staff = pd.merge(target_rentals, staff_df[['staff_id', 'first_name']], on='staff_id')
m_pay = pd.merge(m_staff, payment_df[['rental_id', 'amount']], on='rental_id')

res_46 = m_pay.groupby('first_name')['amount'].sum().reset_index(name='total_revenue')
print(res_46)

  first_name  total_revenue
0        Jon           1.98
1       Mike           3.96


In [49]:
# Soru 47
target_inv = inventory_df[inventory_df['film_id'] == top_film_id]

target_inv_ids = target_inv['inventory_id']
relevant_rentals = rental_df[rental_df['inventory_id'].isin(target_inv_ids)]

res_47 = pd.merge(relevant_rentals, inventory_df[['inventory_id', 'store_id']], on='inventory_id')
res_47 = res_47.groupby('store_id').size().reset_index(name='rental_count')
print(res_47)

   store_id  rental_count
0         1            17
1         2            17


In [50]:
# Soru 48
target_inv = inventory_df[inventory_df['film_id'] == least_film_id]
target_inv_ids = target_inv['inventory_id']

relevant_rentals = rental_df[rental_df['inventory_id'].isin(target_inv_ids)]

res_48 = pd.merge(relevant_rentals, inventory_df[['inventory_id', 'store_id']], on='inventory_id')
res_48 = res_48.groupby('store_id').size().reset_index(name='rental_count')
print(res_48)

   store_id  rental_count
0         1             4


In [51]:
# Soru 49
target_inv_ids = inventory_df[inventory_df['film_id'] == top_rev_film_id]['inventory_id']
target_rentals = rental_df[rental_df['inventory_id'].isin(target_inv_ids)]

m1 = pd.merge(target_rentals, inventory_df[['inventory_id', 'store_id']], on='inventory_id')
m2 = pd.merge(m1, payment_df[['rental_id', 'amount']], on='rental_id')

res_49 = m2.groupby('store_id')['amount'].sum().reset_index(name='total_revenue')
print(res_49)

   store_id  total_revenue
0         1         132.85
1         2          98.88


In [35]:
# Soru 50
s_cols = store_df[['store_id']]
st_cols = staff_df[['staff_id', 'store_id']]
r_cols = rental_df[['rental_id', 'staff_id', 'inventory_id']]
p_cols = payment_df[['rental_id', 'amount']]
i_cols = inventory_df[['inventory_id', 'film_id']]
f_cols = film_df[['film_id', 'title']]

m1 = pd.merge(s_cols, st_cols, on='store_id')
m2 = pd.merge(m1, r_cols, on='staff_id')
m3 = pd.merge(m2, p_cols, on='rental_id')
m4 = pd.merge(m3, i_cols, on='inventory_id')
df = pd.merge(m4, f_cols, on='film_id')

res_50 = df.groupby(['store_id', 'title'])['amount'].sum().reset_index()
res_50 = res_50.sort_values('amount', ascending=True).head(1)
print(res_50)

      store_id              title  amount
1112         2  COMANCHEROS ENEMY    0.99


In [53]:
# Soru 51
c_cols = customer_df[['customer_id', 'first_name', 'last_name']]
r_cols = rental_df[['rental_id', 'customer_id', 'rental_date', 'return_date']]

merged_df = pd.merge(c_cols, r_cols, on='customer_id')

merged_df['rental_date'] = pd.to_datetime(merged_df['rental_date'])
merged_df['return_date'] = pd.to_datetime(merged_df['return_date'])

merged_df = merged_df.dropna(subset=['return_date'])

merged_df['duration'] = (merged_df['return_date'] - merged_df['rental_date']).dt.total_seconds() / 86400

res_51 = merged_df.groupby(['first_name', 'last_name'])['duration'].sum().reset_index(name='total_duration')
res_51 = res_51.sort_values('total_duration', ascending=False)

print(res_51)

    first_name last_name  total_duration
318       KARL      SEAL      264.151389
175    ELEANOR      HUNT      243.095139
105      CLARA      SHAW      235.040278
474     RHONDA   KENNEDY      229.187500
125      DAISY     BATES      221.306250
..         ...       ...             ...
83    CAROLINE    BOWMAN       72.142361
296     JOHNNY    TURPIN       69.217361
33       ANNIE   RUSSELL       64.425694
71       BRIAN     WYMAN       59.231944
550    TIFFANY    JORDAN       59.197917

[599 rows x 3 columns]


In [54]:
# Soru 52
c_cols = category_df[['category_id', 'name']]
fc_cols = film_category_df[['category_id', 'film_id']]
f_cols = film_df[['film_id', 'title']]
i_cols = inventory_df[['inventory_id', 'film_id']]
r_cols = rental_df[['rental_id', 'inventory_id']]

m1 = pd.merge(c_cols, fc_cols, on='category_id')
m2 = pd.merge(m1, f_cols, on='film_id')
m3 = pd.merge(m2, i_cols, on='film_id')
final_df = pd.merge(m3, r_cols, on='inventory_id')

res_52 = final_df.groupby(['name', 'title'])['rental_id'].count().reset_index(name='rental_count')
res_52 = res_52.sort_values('rental_count', ascending=False).head(1)

print(res_52)

       name               title  rental_count
909  Travel  BUCKET BROTHERHOOD            34


In [55]:
# Soru 53
res_53 = final_df.groupby(['name', 'title'])['rental_id'].count().reset_index(name='rental_count')
res_53 = res_53.sort_values('rental_count', ascending=True).head(1)

print(res_53)

            name           title  rental_count
315  Documentary  HARDLY ROBBERS             4


In [56]:
# Soru 54
c_cols = customer_df[['customer_id', 'first_name', 'last_name']]
r_cols = rental_df[['rental_id', 'customer_id']]
p_cols = payment_df[['rental_id', 'amount']]

m1 = pd.merge(c_cols, r_cols, on='customer_id')
final_df = pd.merge(m1, p_cols, on='rental_id')

res_54 = final_df.groupby(['first_name', 'last_name'])['amount'].sum().reset_index(name='total_payment')
res_54 = res_54.sort_values('total_payment', ascending=False)

print(res_54)

    first_name last_name  total_payment
318       KARL      SEAL         221.55
175    ELEANOR      HUNT         216.54
105      CLARA      SHAW         195.58
474     RHONDA   KENNEDY         194.61
389     MARION    SNYDER         194.61
..         ...       ...            ...
33       ANNIE   RUSSELL          58.82
296     JOHNNY    TURPIN          57.81
71       BRIAN     WYMAN          52.88
351      LEONA    OBRIEN          50.86
83    CAROLINE    BOWMAN          50.85

[599 rows x 3 columns]


In [59]:
#Soru 55
f_cols = film_df[['film_id', 'title']]
i_cols = inventory_df[['inventory_id', 'film_id']]
r_cols = rental_df[['rental_id', 'inventory_id', 'rental_date', 'return_date']]

m1 = pd.merge(f_cols, i_cols, on='film_id')
merged_df = pd.merge(m1, r_cols, on='inventory_id')

# Tarih dönüşümü
merged_df['rental_date'] = pd.to_datetime(merged_df['rental_date'])
merged_df['return_date'] = pd.to_datetime(merged_df['return_date'])

# Süre hesapla
merged_df['days'] = (merged_df['return_date'] - merged_df['rental_date']).dt.total_seconds() / 86400

res_55 = merged_df.groupby('title')['days'].max().reset_index(name='max_rental_days')
res_55 = res_55.sort_values('max_rental_days', ascending=False).head(10)

print(res_55)

                  title  max_rental_days
400  HOLOCAUST HIGHBALL         9.249306
393     HIGHBALL POTTER         9.249306
160   CONEHEADS SMOOCHY         9.248611
625          PANIC CLUB         9.248611
535          MASK PEACH         9.248611
868        TRAMP OTHERS         9.248611
602   NOTORIOUS REUNION         9.247222
38         ATTACKS HATE         9.247222
863         TRACY CIDER         9.246528
461        JERSEY SASSY         9.246528


In [60]:
# Soru 56
res_56 = merged_df.groupby('title')['days'].min().reset_index(name='min_rental_days')
res_56 = res_56.sort_values('min_rental_days', ascending=True).head(10)

print(res_56)

                   title  min_rental_days
216      DISCIPLE MOTHER         0.750000
185    CURTAIN VIDEOTAPE         0.750000
701         ROAD ROXANNE         0.750694
815       STRAIGHT HOURS         0.751389
714  RUGRATS SHAKESPEARE         0.751389
296       FIDELITY DEVIL         0.751389
678            RACER EGG         0.751389
475        KISSING DOLLS         0.751389
387       HEAVEN FREEDOM         0.751389
748   SHAKESPEARE SADDLE         0.752083


In [62]:
# Soru 56
c_cols = customer_df[['customer_id', 'first_name', 'last_name']]
r_cols = rental_df[['rental_id', 'customer_id']]
p_cols = payment_df[['rental_id', 'amount']]

m1 = pd.merge(c_cols, r_cols, on='customer_id')
final_df = pd.merge(m1, p_cols, on='rental_id')

res_57 = final_df.groupby(['first_name', 'last_name'])['amount'].mean().reset_index(name='avg_payment')
res_57 = res_57.sort_values('avg_payment', ascending=False)

print(res_57)

    first_name last_name  avg_payment
72    BRITTANY     RILEY     5.704286
154        DON      BONE     5.350000
331      KEVIN   SCHULER     5.308182
348       LENA    JENSEN     5.271250
364     LONNIE    TIRADO     5.267778
..         ...       ...          ...
589      WENDY  HARRISON     3.056667
309     JUDITH       COX     3.050606
62       BOBBY  BOUDREAU     3.047143
296     JOHNNY    TURPIN     3.042632
401     MATTIE   HOFFMAN     2.944545

[599 rows x 3 columns]


In [63]:
# Soru 58
res_58 = merged_df.groupby('title').agg(
    avg_rental_days=('days', 'mean'),
    rental_count=('rental_id', 'count')
).reset_index()

res_58 = res_58.sort_values('rental_count', ascending=False).head(10)

print(res_58[['title', 'avg_rental_days']])

                   title  avg_rental_days
96    BUCKET BROTHERHOOD         4.948448
705     ROCKETEER MOTHER         5.386069
697  RIDGEMONT SUBMARINE         6.010909
361       GRIT CLOCKWORK         5.340864
465       JUGGLER HARDLY         5.509901
312       FORWARD TEMPLE         5.595247
733        SCALAWAG DUCK         4.069640
957            ZORRO ARK         4.539180
853       TIMBERLAND SKY         5.811358
29         APACHE DIVINE         4.269064


In [64]:
# Soru 59
stats = merged_df.groupby('title').agg(
    avg_rental_days=('days', 'mean'),
    rental_count=('rental_id', 'count')
).reset_index()

res_59 = stats.sort_values('rental_count', ascending=True).head(10)

print(res_59[['title', 'avg_rental_days']])

                  title  avg_rental_days
558         MIXED DOORS         5.953299
866         TRAIN BUNCH         3.349306
378      HARDLY ROBBERS         6.948437
585  MUSSOLINI SPOILERS         4.038611
315   FREEDOM CLEOPATRA         2.959722
669        PRIVATE DROP         4.949583
293        FEVER EMPIRE         5.755694
168   CONSPIRACY SPIRIT         4.010000
532     MANNEQUIN WORST         4.414167
435     INFORMER DOUBLE         3.394583


In [65]:
# Soru 60
cust_stats = final_df.groupby(['first_name', 'last_name']).agg(
    avg_payment=('amount', 'mean'),
    total_spent=('amount', 'sum')
).reset_index()

res_60 = cust_stats.sort_values('total_spent', ascending=False).head(10)

print(res_60[['first_name', 'last_name', 'avg_payment']])

    first_name last_name  avg_payment
318       KARL      SEAL     4.923333
175    ELEANOR      HUNT     4.707391
105      CLARA      SHAW     4.656667
474     RHONDA   KENNEDY     4.990000
389     MARION    SNYDER     4.990000
556      TOMMY   COLLAZO     4.911053
590     WESLEY      BULL     4.440000
551        TIM      CARY     4.502821
379     MARCIA      DEAN     4.180476
21         ANA   BRADLEY     5.137059


In [66]:
# Soru 61
c_cols = customer_df[['customer_id', 'first_name', 'last_name']]
r_cols = rental_df[['rental_id', 'customer_id']]
p_cols = payment_df[['rental_id', 'amount']]

m1 = pd.merge(c_cols, r_cols, on='customer_id')
final_df = pd.merge(m1, p_cols, on='rental_id')

stats = final_df.groupby(['first_name', 'last_name']).agg(
    avg_payment=('amount', 'mean'),
    total_spent=('amount', 'sum')
).reset_index()

res_61 = stats.sort_values('total_spent', ascending=True).head(10)
print(res_61[['first_name', 'last_name', 'avg_payment']])

    first_name last_name  avg_payment
83    CAROLINE    BOWMAN     3.390000
351      LEONA    OBRIEN     3.632857
71       BRIAN     WYMAN     4.406667
296     JOHNNY    TURPIN     3.042632
33       ANNIE   RUSSELL     3.267778
319  KATHERINE    RIVERA     4.204286
550    TIFFANY    JORDAN     4.275714
28       ANITA   MORALES     4.190000
401     MATTIE   HOFFMAN     2.944545
334       KIRK   STCLAIR     3.411053


In [67]:
# Soru 62
s_cols = store_df[['store_id']]
st_cols = staff_df[['staff_id', 'store_id']]
r_cols = rental_df[['rental_id', 'staff_id', 'rental_date', 'return_date']]

m1 = pd.merge(s_cols, st_cols, on='store_id')
merged_df = pd.merge(m1, r_cols, on='staff_id')
merged_df['rental_date'] = pd.to_datetime(merged_df['rental_date'])
merged_df['return_date'] = pd.to_datetime(merged_df['return_date'])
merged_df = merged_df.dropna(subset=['return_date'])
merged_df['duration'] = (merged_df['return_date'] - merged_df['rental_date']).dt.total_seconds() / 86400

res_62 = merged_df.groupby('store_id')['duration'].sum().reset_index(name='total_rental_days')
print(res_62)

   store_id  total_rental_days
0         1       39878.755556
1         2       39828.009028


In [68]:
# Soru 63
f_cols = film_df[['film_id', 'title']]
i_cols = inventory_df[['inventory_id', 'film_id']]
r_cols = rental_df[['rental_id', 'inventory_id', 'staff_id', 'rental_date', 'return_date']]
st_cols = staff_df[['staff_id', 'store_id']]

m1 = pd.merge(f_cols, i_cols, on='film_id')
m2 = pd.merge(m1, r_cols, on='inventory_id')
merged_df = pd.merge(m2, st_cols, on='staff_id')

merged_df['rental_date'] = pd.to_datetime(merged_df['rental_date'])
merged_df['return_date'] = pd.to_datetime(merged_df['return_date'])
merged_df['days'] = (merged_df['return_date'] - merged_df['rental_date']).dt.total_seconds() / 86400

res_63 = merged_df.sort_values('days', ascending=False).head(1)
print(res_63[['title', 'store_id', 'days']])

                    title  store_id      days
11418  HOLOCAUST HIGHBALL         1  9.249306


In [69]:
# Soru 64
res_64 = merged_df.sort_values('days', ascending=True).head(1)
print(res_64[['title', 'store_id', 'days']])

                  title  store_id  days
1546  CURTAIN VIDEOTAPE         2  0.75


In [71]:
# Soru 65
res_65 = merged_df.groupby('title')['days'].mean().reset_index(name='avg_rental_days')
print(res_65)

                 title  avg_rental_days
0     ACADEMY DINOSAUR         4.997128
1       ACE GOLDFINGER         5.642708
2     ADAPTATION HOLES         3.465625
3     AFFAIR PREJUDICE         4.758333
4          AFRICAN EGG         7.106629
..                 ...              ...
953     YOUNG LANGUAGE         4.624405
954         YOUTH KICK         5.562963
955       ZHIVAGO CORE         5.950087
956  ZOOLANDER FICTION         5.542647
957          ZORRO ARK         4.539180

[958 rows x 2 columns]


In [72]:
# Soru 66
c_cols = category_df[['category_id', 'name']]
fc_cols = film_category_df[['category_id', 'film_id']]
f_cols = film_df[['film_id', 'title']]
i_cols = inventory_df[['inventory_id', 'film_id']]
r_cols = rental_df[['rental_id', 'inventory_id']]

m1 = pd.merge(c_cols, fc_cols, on='category_id')
m2 = pd.merge(m1, f_cols, on='film_id')
m3 = pd.merge(m2, i_cols, on='film_id')
final_df = pd.merge(m3, r_cols, on='inventory_id')

res_66 = final_df.groupby(['name', 'title'])['rental_id'].count().reset_index(name='rental_count')
res_66 = res_66.sort_values('rental_count', ascending=False)

print(res_66)

            name                title  rental_count
909       Travel   BUCKET BROTHERHOOD            34
536      Foreign     ROCKETEER MOTHER            33
757          New  RIDGEMONT SUBMARINE            32
572        Games       GRIT CLOCKWORK            32
93     Animation       JUGGLER HARDLY            32
..           ...                  ...           ...
513      Foreign      INFORMER DOUBLE             5
570        Games          GLORY TRACY             5
656       Horror          TRAIN BUNCH             4
525      Foreign          MIXED DOORS             4
315  Documentary       HARDLY ROBBERS             4

[958 rows x 3 columns]


In [73]:
# Soru 67
res_67 = res_66.sort_values('rental_count', ascending=True)
print(res_67)

            name                title  rental_count
315  Documentary       HARDLY ROBBERS             4
525      Foreign          MIXED DOORS             4
656       Horror          TRAIN BUNCH             4
593        Games         PRIVATE DROP             5
254       Comedy    FREEDOM CLEOPATRA             5
..           ...                  ...           ...
93     Animation       JUGGLER HARDLY            32
572        Games       GRIT CLOCKWORK            32
757          New  RIDGEMONT SUBMARINE            32
536      Foreign     ROCKETEER MOTHER            33
909       Travel   BUCKET BROTHERHOOD            34

[958 rows x 3 columns]


In [74]:
# Soru 68
s_cols = store_df[['store_id']]
st_cols = staff_df[['staff_id', 'store_id']]
r_cols = rental_df[['rental_id', 'staff_id']]

m1 = pd.merge(s_cols, st_cols, on='store_id')
final_df = pd.merge(m1, r_cols, on='staff_id')

res_68 = final_df.groupby('store_id')['rental_id'].count().reset_index(name='rental_count')
res_68 = res_68.sort_values('rental_count', ascending=False)

print(res_68)

   store_id  rental_count
0         1          8040
1         2          8004


In [75]:
# Soru 69
res_69 = res_68.sort_values('rental_count', ascending=True)
print(res_69)

   store_id  rental_count
1         2          8004
0         1          8040


In [76]:
# Soru 70
a_cols = actor_df[['actor_id', 'first_name', 'last_name']]
fa_cols = film_actor_df[['actor_id', 'film_id']]

merged_df = pd.merge(a_cols, fa_cols, on='actor_id')

res_70 = merged_df.groupby(['first_name', 'last_name'])['film_id'].count().reset_index(name='film_count')
res_70 = res_70.sort_values('film_count', ascending=False)

print(res_70)

    first_name  last_name  film_count
180      SUSAN      DAVIS          54
65        GINA  DEGENERES          42
190     WALTER       TORN          41
123       MARY     KEITEL          40
125    MATTHEW     CARREY          39
..         ...        ...         ...
177      SISSY   SOBIESKI          18
103      JULIA  ZELLWEGER          16
101      JULIA    FAWCETT          15
99        JUDY       DEAN          15
51       EMILY        DEE          14

[199 rows x 3 columns]


In [77]:
# Soru 71
res_71 = res_70.sort_values('film_count', ascending=True)
print(res_71)

    first_name  last_name  film_count
51       EMILY        DEE          14
99        JUDY       DEAN          15
101      JULIA    FAWCETT          15
103      JULIA  ZELLWEGER          16
177      SISSY   SOBIESKI          18
..         ...        ...         ...
125    MATTHEW     CARREY          39
123       MARY     KEITEL          40
190     WALTER       TORN          41
65        GINA  DEGENERES          42
180      SUSAN      DAVIS          54

[199 rows x 3 columns]


In [78]:
# Soru 72
a_cols = actor_df[['actor_id', 'first_name', 'last_name']]
fa_cols = film_actor_df[['actor_id', 'film_id']]
f_cols = film_df[['film_id']] 
i_cols = inventory_df[['inventory_id', 'film_id']]
r_cols = rental_df[['rental_id', 'inventory_id']]

m1 = pd.merge(a_cols, fa_cols, on='actor_id')
m2 = pd.merge(m1, f_cols, on='film_id')
m3 = pd.merge(m2, i_cols, on='film_id')
final_df = pd.merge(m3, r_cols, on='inventory_id')

res_72 = final_df.groupby(['first_name', 'last_name'])['rental_id'].count().reset_index(name='rental_count')
res_72 = res_72.sort_values('rental_count', ascending=False)

print(res_72)

    first_name    last_name  rental_count
180      SUSAN        DAVIS           825
65        GINA    DEGENERES           753
125    MATTHEW       CARREY           678
123       MARY       KEITEL           674
8       ANGELA  WITHERSPOON           654
..         ...          ...           ...
101      JULIA      FAWCETT           255
99        JUDY         DEAN           255
177      SISSY     SOBIESKI           235
103      JULIA    ZELLWEGER           221
51       EMILY          DEE           216

[199 rows x 3 columns]


In [79]:
# Soru 73
a_cols = actor_df[['actor_id', 'first_name', 'last_name']]
fa_cols = film_actor_df[['actor_id', 'film_id']]
i_cols = inventory_df[['inventory_id', 'film_id']]
r_cols = rental_df[['rental_id', 'inventory_id']]

m1 = pd.merge(a_cols, fa_cols, on='actor_id')
m2 = pd.merge(m1, i_cols, on='film_id')
final_df = pd.merge(m2, r_cols, on='inventory_id')

res_73 = final_df.groupby(['first_name', 'last_name'])['rental_id'].count().reset_index(name='rental_count')
res_73 = res_73.sort_values('rental_count', ascending=True)

print(res_73)

    first_name    last_name  rental_count
51       EMILY          DEE           216
103      JULIA    ZELLWEGER           221
177      SISSY     SOBIESKI           235
99        JUDY         DEAN           255
101      JULIA      FAWCETT           255
..         ...          ...           ...
8       ANGELA  WITHERSPOON           654
123       MARY       KEITEL           674
125    MATTHEW       CARREY           678
65        GINA    DEGENERES           753
180      SUSAN        DAVIS           825

[199 rows x 3 columns]


In [81]:
# Soru 74
c_cols = category_df[['category_id', 'name']]
fc_cols = film_category_df[['category_id', 'film_id']]
fa_cols = film_actor_df[['film_id', 'actor_id']]

m1 = pd.merge(c_cols, fc_cols, on='category_id')
final_df = pd.merge(m1, fa_cols, on='film_id')

res_74 = final_df.groupby('name')['actor_id'].nunique().reset_index(name='actor_count')
res_74 = res_74.sort_values('actor_count', ascending=False)

print(res_74)

           name  actor_count
14       Sports          182
8       Foreign          175
12          New          169
5   Documentary          168
13       Sci-Fi          167
0        Action          166
1     Animation          166
15       Travel          166
7        Family          164
2      Children          163
3      Classics          162
6         Drama          162
10       Horror          156
9         Games          150
4        Comedy          147
11        Music          144


In [82]:
# Soru 75
res_75 = res_74.sort_values('actor_count', ascending=True)
print(res_75)

           name  actor_count
11        Music          144
4        Comedy          147
9         Games          150
10       Horror          156
3      Classics          162
6         Drama          162
2      Children          163
7        Family          164
0        Action          166
1     Animation          166
15       Travel          166
13       Sci-Fi          167
5   Documentary          168
12          New          169
8       Foreign          175
14       Sports          182


In [83]:
# Soru 76
c_cols = category_df[['category_id', 'name']]
fc_cols = film_category_df[['category_id', 'film_id']]
i_cols = inventory_df[['inventory_id', 'film_id']]
r_cols = rental_df[['rental_id', 'inventory_id']]

m1 = pd.merge(c_cols, fc_cols, on='category_id')
m2 = pd.merge(m1, i_cols, on='film_id')
final_df = pd.merge(m2, r_cols, on='inventory_id')

res_76 = final_df.groupby('name')['rental_id'].count().reset_index(name='rental_count')
res_76 = res_76.sort_values('rental_count', ascending=False)

print(res_76)

           name  rental_count
14       Sports          1179
1     Animation          1166
0        Action          1112
13       Sci-Fi          1101
7        Family          1096
6         Drama          1060
5   Documentary          1050
8       Foreign          1033
9         Games           969
2      Children           945
4        Comedy           941
12          New           940
3      Classics           939
10       Horror           846
15       Travel           837
11        Music           830


In [84]:
# Soru 77
res_77 = res_76.sort_values('rental_count', ascending=True)
print(res_77)

           name  rental_count
11        Music           830
15       Travel           837
10       Horror           846
3      Classics           939
12          New           940
4        Comedy           941
2      Children           945
9         Games           969
8       Foreign          1033
5   Documentary          1050
6         Drama          1060
7        Family          1096
13       Sci-Fi          1101
0        Action          1112
1     Animation          1166
14       Sports          1179


In [85]:
# Soru 78
p_cols = payment_df[['rental_id', 'amount']]

full_df = pd.merge(final_df, p_cols, on='rental_id')

res_78 = full_df.groupby('name')['amount'].sum().reset_index(name='total_revenue')
res_78 = res_78.sort_values('total_revenue', ascending=False)

print(res_78)

           name  total_revenue
14       Sports        5314.21
13       Sci-Fi        4756.98
1     Animation        4656.30
6         Drama        4587.39
4        Comedy        4383.58
0        Action        4375.85
12          New        4351.62
9         Games        4281.33
8       Foreign        4270.67
7        Family        4226.07
5   Documentary        4217.52
10       Horror        3722.54
2      Children        3655.55
3      Classics        3639.59
15       Travel        3549.64
11        Music        3417.72


In [86]:
# Soru 79
res_79 = res_78.sort_values('total_revenue', ascending=True)
print(res_79)

           name  total_revenue
11        Music        3417.72
15       Travel        3549.64
3      Classics        3639.59
2      Children        3655.55
10       Horror        3722.54
5   Documentary        4217.52
7        Family        4226.07
8       Foreign        4270.67
9         Games        4281.33
12          New        4351.62
0        Action        4375.85
4        Comedy        4383.58
6         Drama        4587.39
1     Animation        4656.30
13       Sci-Fi        4756.98
14       Sports        5314.21


In [87]:
# Soru 80
c_cols = customer_df[['customer_id', 'first_name', 'last_name']]
r_cols = rental_df[['rental_id', 'customer_id']]
p_cols = payment_df[['rental_id', 'amount']]

m1 = pd.merge(c_cols, r_cols, on='customer_id')
final_df = pd.merge(m1, p_cols, on='rental_id')

stats = final_df.groupby(['first_name', 'last_name']).agg(
    rental_count=('rental_id', 'count'),
    avg_payment=('amount', 'mean')
).reset_index()

res_80 = stats.sort_values('rental_count', ascending=False).head(10)

print(res_80[['first_name', 'last_name', 'avg_payment']])

    first_name last_name  avg_payment
175    ELEANOR      HUNT     4.707391
318       KARL      SEAL     4.923333
379     MARCIA      DEAN     4.180476
105      CLARA      SHAW     4.656667
536      TAMMY   SANDERS     3.794878
590     WESLEY      BULL     4.440000
531        SUE    PETERS     3.865000
389     MARION    SNYDER     4.990000
551        TIM      CARY     4.502821
474     RHONDA   KENNEDY     4.990000


In [88]:
#Soru 81
res_81 = stats.sort_values('rental_count', ascending=True).head(10)
print(res_81[['first_name', 'last_name', 'avg_payment']])

    first_name last_name  avg_payment
71       BRIAN     WYMAN     4.406667
550    TIFFANY    JORDAN     4.275714
351      LEONA    OBRIEN     3.632857
319  KATHERINE    RIVERA     4.204286
83    CAROLINE    BOWMAN     3.390000
28       ANITA   MORALES     4.190000
277     JEROME    KENYON     4.615000
356     LESTER     KRAUS     4.115000
290      JOANN   GARDNER     4.177500
35     ANTONIO      MEEK     4.927500


In [89]:
# Soru 82
s_cols = store_df[['store_id']]
st_cols = staff_df[['staff_id', 'store_id']]
r_cols = rental_df[['rental_id', 'staff_id', 'inventory_id']]
i_cols = inventory_df[['inventory_id', 'film_id']]
fc_cols = film_category_df[['film_id', 'category_id']]
c_cols = category_df[['category_id', 'name']]

m1 = pd.merge(s_cols, st_cols, on='store_id')
m2 = pd.merge(m1, r_cols, on='staff_id')

m3 = pd.merge(m2, i_cols, on='inventory_id')
m4 = pd.merge(m3, fc_cols, on='film_id')
final_df = pd.merge(m4, c_cols, on='category_id')

res_82 = final_df.groupby(['store_id', 'name'])['rental_id'].count().reset_index(name='rental_count')
res_82 = res_82.sort_values('rental_count', ascending=False)

print(res_82)

    store_id         name  rental_count
30         2       Sports           614
17         2    Animation           584
1          1    Animation           582
7          1       Family           565
14         1       Sports           565
0          1       Action           562
13         1       Sci-Fi           553
16         2       Action           550
29         2       Sci-Fi           548
24         2      Foreign           543
22         2        Drama           532
23         2       Family           531
6          1        Drama           528
5          1  Documentary           525
21         2  Documentary           525
9          1        Games           494
8          1      Foreign           490
12         1          New           489
2          1     Children           483
25         2        Games           475
4          1       Comedy           475
3          1     Classics           475
20         2       Comedy           466
19         2     Classics           464


In [90]:
# Soru 83
res_83 = res_82.sort_values('rental_count', ascending=True)
print(res_83)

    store_id         name  rental_count
27         2        Music           398
15         1       Travel           399
10         1       Horror           423
26         2       Horror           423
11         1        Music           432
31         2       Travel           438
28         2          New           451
18         2     Children           462
19         2     Classics           464
20         2       Comedy           466
3          1     Classics           475
4          1       Comedy           475
25         2        Games           475
2          1     Children           483
12         1          New           489
8          1      Foreign           490
9          1        Games           494
5          1  Documentary           525
21         2  Documentary           525
6          1        Drama           528
23         2       Family           531
22         2        Drama           532
24         2      Foreign           543
29         2       Sci-Fi           548


In [91]:
# Soru 84
s_cols = store_df[['store_id']]
st_cols = staff_df[['staff_id', 'store_id']]
r_cols = rental_df[['rental_id', 'staff_id']]

m1 = pd.merge(s_cols, st_cols, on='store_id')
final_df = pd.merge(m1, r_cols, on='staff_id')

res_84 = final_df.groupby('store_id')['rental_id'].count().reset_index(name='total_rentals')
res_84 = res_84.sort_values('total_rentals', ascending=False).head(1)

print(res_84)

   store_id  total_rentals
0         1           8040


In [103]:
# Soru 85
s_cols = store_df[['store_id']]
st_cols = staff_df[['staff_id', 'store_id']]
r_cols = rental_df[['rental_id', 'staff_id']]

m1 = pd.merge(s_cols, st_cols, on='store_id')
final_df = pd.merge(m1, r_cols, on='staff_id')

res_85 = final_df.groupby('store_id')['rental_id'].count().reset_index(name='total_rentals')
res_85 = res_85.sort_values('total_rentals', ascending=True).head(1)

print(res_85)

   store_id  total_rentals
1         2           8004


In [95]:
# Soru 86
c_cols = customer_df[['customer_id', 'first_name', 'last_name']]
r_cols = rental_df[['rental_id', 'customer_id']]

final_df = pd.merge(c_cols, r_cols, on='customer_id')

res_86 = final_df.groupby(['first_name', 'last_name'])['rental_id'].count().reset_index(name='total_rentals')
res_86 = res_86.sort_values('total_rentals', ascending=False).head(1)

print(res_86)

    first_name last_name  total_rentals
175    ELEANOR      HUNT             46


In [104]:
#Soru 87
c_cols = customer_df[['customer_id', 'first_name', 'last_name']]
r_cols = rental_df[['rental_id', 'customer_id']]

final_df = pd.merge(c_cols, r_cols, on='customer_id')

res_87 = final_df.groupby(['first_name', 'last_name'])['rental_id'].count().reset_index(name='total_rentals')
res_87 = res_87.sort_values('total_rentals', ascending=True).head(1)

print(res_87)

   first_name last_name  total_rentals
71      BRIAN     WYMAN             12


In [97]:
# Soru 88
f_cols = film_df[['film_id', 'title']]
i_cols = inventory_df[['inventory_id', 'film_id']]
r_cols = rental_df[['rental_id', 'inventory_id']]

m1 = pd.merge(f_cols, i_cols, on='film_id')
final_df = pd.merge(m1, r_cols, on='inventory_id')

res_88 = final_df.groupby('title')['rental_id'].count().reset_index(name='total_rentals')
res_88 = res_88.sort_values('total_rentals', ascending=False).head(1)

print(res_88)

                 title  total_rentals
96  BUCKET BROTHERHOOD             34


In [105]:
# Soru 89
f_cols = film_df[['film_id', 'title']]
i_cols = inventory_df[['inventory_id', 'film_id']]
r_cols = rental_df[['rental_id', 'inventory_id']]

m1 = pd.merge(f_cols, i_cols, on='film_id')
final_df = pd.merge(m1, r_cols, on='inventory_id')

res_89 = final_df.groupby('title')['rental_id'].count().reset_index(name='total_rentals')
res_89 = res_89.sort_values('total_rentals', ascending=True).head(1)

print(res_89)

           title  total_rentals
558  MIXED DOORS              4


In [99]:
# Soru 90
c_cols = customer_df[['customer_id', 'first_name', 'last_name']]
r_cols = rental_df[['rental_id', 'customer_id']]
p_cols = payment_df[['rental_id', 'amount']]

m1 = pd.merge(c_cols, r_cols, on='customer_id')
final_df = pd.merge(m1, p_cols, on='rental_id')


res_90 = final_df.groupby(['first_name', 'last_name'])['amount'].sum().reset_index(name='total_revenue')
res_90 = res_90.sort_values('total_revenue', ascending=False).head(1)

print(res_90)

    first_name last_name  total_revenue
318       KARL      SEAL         221.55


In [106]:
# Soru 91
c_cols = customer_df[['customer_id', 'first_name', 'last_name']]
r_cols = rental_df[['rental_id', 'customer_id']]
p_cols = payment_df[['rental_id', 'amount']]

m1 = pd.merge(c_cols, r_cols, on='customer_id')
final_df = pd.merge(m1, p_cols, on='rental_id')

res_91 = final_df.groupby(['first_name', 'last_name'])['amount'].sum().reset_index(name='total_revenue')
res_91 = res_91.sort_values('total_revenue', ascending=True).head(1)

print(res_91)

   first_name last_name  total_revenue
83   CAROLINE    BOWMAN          50.85


In [101]:
# Soru 92
f_cols = film_df[['film_id', 'title']]
fc_cols = film_category_df[['film_id', 'category_id']]
c_cols = category_df[['category_id', 'name']]
i_cols = inventory_df[['film_id', 'inventory_id']]
r_cols = rental_df[['inventory_id', 'rental_id']]
p_cols = payment_df[['rental_id', 'amount']]

m1 = pd.merge(f_cols, fc_cols, on='film_id')
m2 = pd.merge(m1, c_cols, on='category_id')

m3 = pd.merge(m2, i_cols, on='film_id')
m4 = pd.merge(m3, r_cols, on='inventory_id')
final_df = pd.merge(m4, p_cols, on='rental_id')

res_92 = final_df.groupby(['title', 'name'])['amount'].sum().reset_index(name='total_revenue')
res_92 = res_92.sort_values('total_revenue', ascending=False).head(1)

print(res_92)

                title   name  total_revenue
841  TELEGRAPH VOYAGE  Music         231.73


In [107]:
# Soru 93
f_cols = film_df[['film_id', 'title']]
fc_cols = film_category_df[['film_id', 'category_id']]
c_cols = category_df[['category_id', 'name']]
i_cols = inventory_df[['film_id', 'inventory_id']]
r_cols = rental_df[['inventory_id', 'rental_id']]
p_cols = payment_df[['rental_id', 'amount']]

m1 = pd.merge(f_cols, fc_cols, on='film_id')
m2 = pd.merge(m1, c_cols, on='category_id')
m3 = pd.merge(m2, i_cols, on='film_id')
m4 = pd.merge(m3, r_cols, on='inventory_id')
final_df = pd.merge(m4, p_cols, on='rental_id')

res_93 = final_df.groupby(['title', 'name'])['amount'].sum().reset_index(name='total_revenue')
res_93 = res_93.sort_values('total_revenue', ascending=True).head(1)

print(res_93)

           title    name  total_revenue
847  TEXAS WATCH  Horror           5.94


In [108]:
# Soru 94
s_cols = store_df[['store_id']]
st_cols = staff_df[['store_id', 'staff_id']]
r_cols = rental_df[['staff_id', 'rental_id']]
p_cols = payment_df[['rental_id', 'amount']]

m1 = pd.merge(s_cols, st_cols, on='store_id')
m2 = pd.merge(m1, r_cols, on='staff_id')
final_df = pd.merge(m2, p_cols, on='rental_id')

res_94 = final_df.groupby('store_id')['amount'].sum().reset_index(name='total_revenue')
res_94 = res_94.sort_values('total_revenue', ascending=False).head(1)

print(res_94)

   store_id  total_revenue
1         2       33881.94


In [109]:
# Soru 95
res_95 = final_df.groupby('store_id')['amount'].sum().reset_index(name='total_revenue')
res_95 = res_95.sort_values('total_revenue', ascending=True).head(1)

print(res_95)

   store_id  total_revenue
0         1       33524.62


In [111]:
#Soru 96
c_cols = customer_df[['customer_id', 'first_name', 'last_name']]
r_cols = rental_df[['customer_id', 'inventory_id']]
i_cols = inventory_df[['inventory_id', 'film_id']]

m1 = pd.merge(c_cols, r_cols, on='customer_id')
final_df = pd.merge(m1, i_cols, on='inventory_id')

res_96 = final_df.groupby(['first_name', 'last_name'])['film_id'].nunique().reset_index(name='unique_movies')
res_96 = res_96.sort_values('unique_movies', ascending=False).head(1)

print(res_96)

    first_name last_name  unique_movies
175    ELEANOR      HUNT             46


In [116]:
# Soru 97
c_cols = customer_df[['customer_id', 'first_name', 'last_name']]
r_cols = rental_df[['customer_id', 'inventory_id']]
i_cols = inventory_df[['inventory_id', 'film_id']]

m1 = pd.merge(c_cols, r_cols, on='customer_id')
final_df = pd.merge(m1, i_cols, on='inventory_id')

grouped_df = final_df.groupby(['first_name', 'last_name'])['film_id'].nunique().reset_index(name='unique_movies')

res_97 = grouped_df.sort_values('unique_movies', ascending=True).head(1)

print(res_97)

   first_name last_name  unique_movies
71      BRIAN     WYMAN             12


In [113]:
# SORU 98
f_cols = film_df[['film_id', 'title']]
i_cols = inventory_df[['film_id', 'inventory_id']]
r_cols = rental_df[['inventory_id', 'customer_id']]

m1 = pd.merge(f_cols, i_cols, on='film_id')
final_df = pd.merge(m1, r_cols, on='inventory_id')

res_98 = final_df.groupby('title')['customer_id'].nunique().reset_index(name='unique_customers')
res_98 = res_98.sort_values('unique_customers', ascending=False).head(1)

print(res_98)

                 title  unique_customers
96  BUCKET BROTHERHOOD                33


In [117]:
# Soru 99
f_cols = film_df[['film_id', 'title']]
i_cols = inventory_df[['film_id', 'inventory_id']]
r_cols = rental_df[['inventory_id', 'customer_id']]

m1 = pd.merge(f_cols, i_cols, on='film_id')
final_df = pd.merge(m1, r_cols, on='inventory_id')
grouped_df = final_df.groupby('title')['customer_id'].nunique().reset_index(name='unique_customers')
res_99 = grouped_df.sort_values('unique_customers', ascending=True).head(1)

print(res_99)

           title  unique_customers
866  TRAIN BUNCH                 4


In [115]:
# Soru 100
s_cols = store_df[['store_id']]
st_cols = staff_df[['store_id', 'staff_id']]
r_cols = rental_df[['staff_id', 'rental_id', 'inventory_id']]
p_cols = payment_df[['rental_id', 'amount']]
i_cols = inventory_df[['inventory_id', 'film_id']]
fc_cols = film_category_df[['film_id', 'category_id']]
c_cols = category_df[['category_id', 'name']]

m1 = pd.merge(s_cols, st_cols, on='store_id')
m2 = pd.merge(m1, r_cols, on='staff_id')
m3 = pd.merge(m2, p_cols, on='rental_id')

m4 = pd.merge(m3, i_cols, on='inventory_id')
m5 = pd.merge(m4, fc_cols, on='film_id')
final_df = pd.merge(m5, c_cols, on='category_id')

res_100 = final_df.groupby(['store_id', 'name'])['amount'].sum().reset_index(name='total_revenue')
res_100 = res_100.sort_values('total_revenue', ascending=False).head(1)

print(res_100)

    store_id    name  total_revenue
30         2  Sports        2761.85
